# CrashLens GPU Workload Demo

This notebook demonstrates CrashLens with **real NVIDIA GPU metrics**.

**Setup:**
1. Runtime → Change runtime type → **GPU** → Save
2. Run all cells below
3. Check dashboard: https://frontend-zeta-eight-92.vercel.app/dashboard

In [ ]:
# Install CrashLens SDK
!pip install requests -q
!git clone https://github.com/kavinsaravan/ML-Failure-Doctor.git
!cd ML-Failure-Doctor/crashlens-sdk && pip install -e . -q

print("✅ CrashLens SDK installed!")

In [ ]:
# Verify GPU is available
import torch

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU detected. Make sure you selected GPU runtime!")

In [ ]:
# Initialize CrashLens
import sys
sys.path.insert(0, '/content/ML-Failure-Doctor/crashlens-sdk')

from crashlens import WorkloadTracker

tracker = WorkloadTracker("https://invigorating-empathy-production-dee5.up.railway.app")
print("✅ CrashLens tracker initialized!")

## Demo 1: Successful GPU Workload

In [ ]:
with tracker.track("Colab Demo - Successful Training"):
    print("Training simple neural network on GPU...")
    
    # Create model and data
    model = torch.nn.Sequential(
        torch.nn.Linear(1000, 500),
        torch.nn.ReLU(),
        torch.nn.Linear(500, 100)
    ).cuda()
    
    # Training loop
    for epoch in range(10):
        x = torch.randn(32, 1000).cuda()
        y = model(x)
        loss = y.mean()
        print(f"Epoch {epoch+1}/10 - Loss: {loss.item():.4f}")
    
    print("\n✅ Training completed successfully!")
    print("Check dashboard: https://frontend-zeta-eight-92.vercel.app/dashboard")

## Demo 2: GPU Out of Memory Failure

In [ ]:
with tracker.track("Colab Demo - GPU OOM"):
    print("Allocating large tensors...")
    
    # This will fail with OOM on most Colab GPUs
    tensors = []
    for i in range(100):
        # Each tensor is ~1GB
        tensor = torch.randn(128, 1024, 1024).cuda()
        tensors.append(tensor)
        print(f"Allocated tensor {i+1}")

## Demo 3: Missing Dependency Error

In [ ]:
with tracker.track("Colab Demo - Missing Package"):
    print("Importing non-existent package...")
    import totally_fake_package_that_doesnt_exist
    print("This won't print")

## Demo 4: Real PyTorch Training with Failure

This simulates a realistic scenario where batch size is too large.

In [ ]:
with tracker.track("Colab Demo - Batch Size Too Large"):
    print("Training with very large batch size...")
    
    # Create large model
    model = torch.nn.Sequential(
        torch.nn.Linear(10000, 10000),
        torch.nn.ReLU(),
        torch.nn.Linear(10000, 10000),
        torch.nn.ReLU(),
        torch.nn.Linear(10000, 1000)
    ).cuda()
    
    # Batch size way too large - will OOM
    batch_size = 1024
    x = torch.randn(batch_size, 10000).cuda()
    y = model(x)
    print("Completed (shouldn't reach here)")

## View Results

Check your dashboard to see:
- ✅ Real NVIDIA GPU metrics (memory, utilization, temperature)
- ✅ Automatic failure classification
- ✅ AI-powered diagnosis with recommended fixes

**Dashboard:** https://frontend-zeta-eight-92.vercel.app/dashboard